# Value Iteration: solve a finite MDP from its model

Value Iteration repeatedly applies the Bellman optimality backup

$$V_{k+1}(s)=\max_a\sum_{s',r}p(s',r\mid s,a)\left[r+\gamma(1-d)V_k(s')\right].$$

Here $V_k(s)$ is the value of state $s$ after sweep $k$, $p$ is the transition model, $\gamma$ is the discount factor, and $d$ marks terminal transitions. `FrozenLake-v1` exposes this finite model directly, so no sampling or neural network is needed.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

ENV_ID = "FrozenLake-v1"
GAMMA = 0.99
TOLERANCE = 1e-10
MAX_SWEEPS = 1_000

env = gym.make(ENV_ID, is_slippery=True)
model = env.unwrapped.P
state_dim = env.observation_space.n
action_dim = env.action_space.n

## 1. Apply Bellman optimality backups

For each action, sum over every model outcome. Terminal outcomes contribute their immediate reward but no bootstrap value. Stop when the largest value change is below `TOLERANCE`, then extract the greedy policy.

In [ ]:
def action_values(state, values):
    estimates = np.zeros(action_dim)
    for action in range(action_dim):
        for probability, next_state, reward, terminated in model[state][action]:
            bootstrap = 0.0 if terminated else values[next_state]
            estimates[action] += probability * (reward + GAMMA * bootstrap)
    return estimates


def value_iteration():
    values = np.zeros(state_dim)
    residuals = []
    for _ in range(MAX_SWEEPS):
        updated = np.array([action_values(state, values).max() for state in range(state_dim)])
        residual = np.max(np.abs(updated - values))
        residuals.append(residual)
        values = updated
        if residual < TOLERANCE:
            break
    policy = np.array([action_values(state, values).argmax() for state in range(state_dim)])
    return values, policy, residuals


values, policy, residuals = value_iteration()
env.close()
print("Sweeps:", len(residuals))
print("Greedy policy:\n", policy.reshape(4, 4))

## 2. Inspect convergence and state values

The Bellman residual should shrink toward zero. The value map shows the probability-discounted usefulness of each grid position under the optimal policy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogy(residuals)
axes[0].set(xlabel="Sweep", ylabel="Bellman residual", title="Convergence")
image = axes[1].imshow(values.reshape(4, 4), cmap="viridis")
axes[1].set(title="Optimal state values")
fig.colorbar(image, ax=axes[1])
plt.tight_layout()
plt.show()

## 3. Watch the planned policy

This opens a separate rendered environment and follows the greedy policy for 5 episodes.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode="human", is_slippery=True)
episode_returns = []
try:
    for episode in range(5):
        state, _ = evaluation_env.reset(seed=10 + episode)
        total_reward = 0.0
        done = False
        while not done:
            state, reward, terminated, truncated, _ = evaluation_env.step(int(policy[state]))
            total_reward += reward
            done = terminated or truncated
        episode_returns.append(total_reward)
finally:
    evaluation_env.close()

print("Episode returns:", episode_returns)
print(f"Mean return: {np.mean(episode_returns):.2f}")